# 01 — Data Exploration
**Inspect the raw October 2019 event dataset before transformation.**


## Objective
- Define the purpose of this stage.
- Record key decisions and assumptions.
- Keep outputs reproducible and easy to trace to the previous stage.


In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("ECommerceCustomerBehaviorAnalytics-01-Exploration")
    .getOrCreate())
print("Spark version:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/27 23:56:15 WARN Utils: Your hostname, Vaishnavis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.102 instead (on interface en0)
26/09/27 23:56:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/vaishnavipoojary/Desktop/ecommerce-customer-behavior-analytics/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/27 23:56:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where

Spark version: 4.2.0


In [2]:
FILE_PATH = "../data/2019-Oct.csv"
df = spark.read.csv(FILE_PATH, header=True, inferSchema=True)
print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()


[Stage 2:=======================================================> (42 + 1) / 43]

Rows: 42448764
Columns: 9
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [3]:
df.groupBy("event_type") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

[Stage 5:======================================================>  (41 + 2) / 43]

+----------+--------+
|event_type|   count|
+----------+--------+
|      view|40779399|
|      cart|  926516|
|  purchase|  742849|
+----------+--------+



In [4]:
from pyspark.sql.functions import col, sum as spark_sum

df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

[Stage 8:======================================================>  (41 + 2) / 43]

+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|  brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+
|         0|         0|         0|          0|     13515609|6113008|    0|      0|           2|
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+



In [5]:
from pyspark.sql.functions import col, count, when, round

null_summary = df.select([
    round(
        (count(when(col(c).isNull(), c)) / count("*")) * 100,
        2
    ).alias(c)
    for c in df.columns
])

null_summary.show()

[Stage 11:=====================================================>  (41 + 2) / 43]

+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+
|       0.0|       0.0|       0.0|        0.0|        31.84| 14.4|  0.0|    0.0|         0.0|
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+



In [6]:
df.selectExpr(
    "min(event_time) as min_event_time",
    "max(event_time) as max_event_time"
).show(truncate=False)

[Stage 14:=====================================================>  (41 + 2) / 43]

+-------------------+-------------------+
|min_event_time     |max_event_time     |
+-------------------+-------------------+
|2019-10-01 05:30:00|2019-11-01 05:29:59|
+-------------------+-------------------+



In [7]:
df.selectExpr(
    "min(price) as min_price",
    "max(price) as max_price",
    "avg(price) as avg_price",
    "sum(case when price <= 0 then 1 else 0 end) as non_positive_prices"
).show()

[Stage 17:=====================================================>  (41 + 2) / 43]

+---------+---------+-----------------+-------------------+
|min_price|max_price|        avg_price|non_positive_prices|
+---------+---------+-----------------+-------------------+
|      0.0|  2574.07|290.3236606848809|              68673|
+---------+---------+-----------------+-------------------+



In [8]:
from pyspark.sql.functions import col, sum as spark_sum, when

invalid_price_summary = df.select(
    spark_sum(when(col("price") <= 0, 1).otherwise(0)).alias("non_positive_price_rows"),
    spark_sum(when(col("price") == 0, 1).otherwise(0)).alias("zero_price_rows"),
    spark_sum(when(col("price") < 0, 1).otherwise(0)).alias("negative_price_rows")
)

invalid_price_summary.show()

[Stage 20:=====================================================>  (41 + 2) / 43]

+-----------------------+---------------+-------------------+
|non_positive_price_rows|zero_price_rows|negative_price_rows|
+-----------------------+---------------+-------------------+
|                  68673|          68673|                  0|
+-----------------------+---------------+-------------------+



In [9]:
df.filter(col("price") <= 0) \
  .groupBy("event_type") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

[Stage 23:=====================================================>  (41 + 2) / 43]

+----------+-----+
|event_type|count|
+----------+-----+
|      view|68523|
|      cart|  150|
+----------+-----+



In [10]:
from pyspark.sql.functions import countDistinct

cardinality = df.select(
    countDistinct("user_id").alias("unique_users"),
    countDistinct("product_id").alias("unique_products"),
    countDistinct("user_session").alias("unique_sessions"),
    countDistinct("category_id").alias("unique_categories"),
    countDistinct("brand").alias("unique_brands")
)

cardinality.show()

[Stage 28:===============>                                         (3 + 8) / 11]

+------------+---------------+---------------+-----------------+-------------+
|unique_users|unique_products|unique_sessions|unique_categories|unique_brands|
+------------+---------------+---------------+-----------------+-------------+
|     3022290|         166794|        9244421|              624|         3445|
+------------+---------------+---------------+-----------------+-------------+



In [11]:
df.show(10, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 05:30:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 05:30:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-10-01 05:30:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-10-01 05:3

In [12]:
df.filter(col("event_type") == "purchase") \
  .show(10, truncate=False)

+-------------------+----------+----------+-------------------+---------------------------------+-------+------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                    |brand  |price |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+---------------------------------+-------+------+---------+------------------------------------+
|2019-10-01 05:32:14|purchase  |1004856   |2053013555631882655|electronics.smartphone           |samsung|130.76|543272936|8187d148-3c41-46d4-b0c0-9c08cd9dc564|
|2019-10-01 05:34:37|purchase  |1002532   |2053013555631882655|electronics.smartphone           |apple  |642.69|551377651|3c80f0d6-e9ec-4181-8c5c-837a30be2d68|
|2019-10-01 05:36:02|purchase  |5100816   |2053013553375346967|NULL                             |xiaomi |29.51 |514591159|0e5dfc4b-2a55-43e6-8c05-97e1f07fbb56|
|2019-10-01 05:37:07|purchase  |13800054

In [13]:
from pyspark.sql.functions import to_date

daily_events = (
    df.withColumn("event_date", to_date("event_time"))
      .groupBy("event_date", "event_type")
      .count()
      .orderBy("event_date", "event_type")
)

daily_events.show(50, truncate=False)

[Stage 34:=====================================================>  (41 + 2) / 43]

+----------+----------+-------+
|event_date|event_type|count  |
+----------+----------+-------+
|2019-10-01|cart      |15647  |
|2019-10-01|purchase  |18020  |
|2019-10-01|view      |1093514|
|2019-10-02|cart      |17151  |
|2019-10-02|purchase  |19599  |
|2019-10-02|view      |1171419|
|2019-10-03|cart      |17955  |
|2019-10-03|purchase  |18780  |
|2019-10-03|view      |1076506|
|2019-10-04|cart      |43760  |
|2019-10-04|purchase  |26964  |
|2019-10-04|view      |1335512|
|2019-10-05|cart      |35578  |
|2019-10-05|purchase  |23508  |
|2019-10-05|view      |1258869|
|2019-10-06|cart      |33615  |
|2019-10-06|purchase  |22461  |
|2019-10-06|view      |1280437|
|2019-10-07|cart      |18031  |
|2019-10-07|purchase  |21444  |
|2019-10-07|view      |1173274|
|2019-10-08|cart      |18418  |
|2019-10-08|purchase  |22950  |
|2019-10-08|view      |1320852|
|2019-10-09|cart      |18514  |
|2019-10-09|purchase  |22806  |
|2019-10-09|view      |1305174|
|2019-10-10|cart      |18477  |
|2019-10

In [14]:
daily_events.groupBy("event_date") \
    .pivot("event_type") \
    .sum("count") \
    .orderBy("event_date") \
    .show(40)

[Stage 50:=====================================================>  (41 + 2) / 43]

+----------+-----+--------+-------+
|event_date| cart|purchase|   view|
+----------+-----+--------+-------+
|2019-10-01|15647|   18020|1093514|
|2019-10-02|17151|   19599|1171419|
|2019-10-03|17955|   18780|1076506|
|2019-10-04|43760|   26964|1335512|
|2019-10-05|35578|   23508|1258869|
|2019-10-06|33615|   22461|1280437|
|2019-10-07|18031|   21444|1173274|
|2019-10-08|18418|   22950|1320852|
|2019-10-09|18514|   22806|1305174|
|2019-10-10|18477|   21534|1229487|
|2019-10-11|21739|   26381|1445935|
|2019-10-12|20670|   25377|1423101|
|2019-10-13|22847|   29175|1590427|
|2019-10-14|45555|   28756|1394718|
|2019-10-15|49344|   26146|1462800|
|2019-10-16|45375|   31457|1447924|
|2019-10-17|40377|   28266|1336759|
|2019-10-18|40027|   26027|1405094|
|2019-10-19|39401|   24607|1369679|
|2019-10-20|40604|   25075|1458159|
|2019-10-21|37019|   25375|1349302|
|2019-10-22|37142|   25429|1358302|
|2019-10-23|36239|   24664|1320403|
|2019-10-24|35568|   23897|1254055|
|2019-10-25|36754|   24046|1

In [16]:
purchases = df.filter(col("event_type") == "purchase")

In [17]:
print("Purchase events:", purchases.count())

[Stage 56:=====================================================>  (41 + 2) / 43]

Purchase events: 742849


In [18]:
purchases.selectExpr(
    "min(price) as min_purchase_price",
    "max(price) as max_purchase_price",
    "avg(price) as avg_purchase_price",
    "sum(price) as total_purchase_value"
).show()

[Stage 59:=====================================================>  (41 + 2) / 43]

+------------------+------------------+------------------+--------------------+
|min_purchase_price|max_purchase_price|avg_purchase_price|total_purchase_value|
+------------------+------------------+------------------+--------------------+
|              0.77|           2574.07|309.56156940373995|2.2995750226999882E8|
+------------------+------------------+------------------+--------------------+



In [19]:
purchase_daily = (
    purchases
    .withColumn("event_date", to_date("event_time"))
    .groupBy("event_date")
    .agg(
        spark_sum("price").alias("daily_revenue"),
        countDistinct("user_id").alias("daily_purchasing_users")
    )
    .orderBy("event_date")
)

purchase_daily.show(40, truncate=False)

[Stage 62:======================================================> (42 + 1) / 43]

+----------+------------------+----------------------+
|event_date|daily_revenue     |daily_purchasing_users|
+----------+------------------+----------------------+
|2019-10-01|5811546.060000006 |13253                 |
|2019-10-02|6264453.440000014 |13926                 |
|2019-10-03|6057844.230000009 |13415                 |
|2019-10-04|8660174.11000004  |19078                 |
|2019-10-05|7330970.36000002  |16769                 |
|2019-10-06|6826011.130000013 |16239                 |
|2019-10-07|6372978.829999997 |15363                 |
|2019-10-08|6754188.640000012 |16585                 |
|2019-10-09|6876760.780000006 |16349                 |
|2019-10-10|6531304.25999999  |15681                 |
|2019-10-11|7788623.929999985 |18758                 |
|2019-10-12|7305593.199999988 |18419                 |
|2019-10-13|8193613.950000005 |21149                 |
|2019-10-14|9532674.900000012 |19487                 |
|2019-10-15|8552300.69        |18413                 |
|2019-10-1

In [20]:
user_event_activity = (
    df.groupBy("event_type")
      .agg(
          countDistinct("user_id").alias("unique_users")
      )
      .orderBy("unique_users", ascending=False)
)

user_event_activity.show()

[Stage 70:>                                                       (0 + 10) / 11]

+----------+------------+
|event_type|unique_users|
+----------+------------+
|      view|     3022130|
|  purchase|      347118|
|      cart|      337117|
+----------+------------+



In [21]:
purchase_users = purchases.select("user_id").distinct()

print("Unique purchasing users:", purchase_users.count())

[Stage 74:=====================================================>  (41 + 2) / 43]

Unique purchasing users: 347118


In [22]:
user_activity = (
    df.groupBy("user_id")
      .agg(
          spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("views"),
          spark_sum(when(col("event_type") == "cart", 1).otherwise(0)).alias("carts"),
          spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("purchases")
      )
)

user_activity.orderBy(col("purchases").desc()).show(20)

[Stage 82:>                                                       (0 + 10) / 11]

+---------+-----+-----+---------+
|  user_id|views|carts|purchases|
+---------+-----+-----+---------+
|512386086|  630|  167|      322|
|523974502|  452|  224|      291|
|513320236|  588|  122|      242|
|517728689|  611|  494|      204|
|543312954|  458|  105|      193|
|513117637|  466|   56|      185|
|519267944|  315|  184|      183|
|530834332|  489|   52|      170|
|541510103|  345|  207|      165|
|551557821|  194|   89|      154|
|549109608|  351|  200|      154|
|513322839|  287|   70|      146|
|546967331|  325|  124|      141|
|556747652|  424|   37|      139|
|519322381|  465|   94|      136|
|558420737|  480|   67|      132|
|547330965|  563|   35|      131|
|547101738|  498|  133|      130|
|525351013| 1047|  143|      129|
|551361102|  563|  140|      127|
+---------+-----+-----+---------+
only showing top 20 rows


In [23]:
session_summary = (
    df.groupBy("user_session")
      .agg(
          countDistinct("user_id").alias("users"),
          count("*").alias("events"),
          spark_sum(
              when(col("event_type") == "purchase", 1).otherwise(0)
          ).alias("purchase_events")
      )
)

session_summary.show(20, truncate=False)

[Stage 85:==================================================>     (10 + 1) / 11]

+------------------------------------+-----+------+---------------+
|user_session                        |users|events|purchase_events|
+------------------------------------+-----+------+---------------+
|cf859590-0ff5-4619-8a6b-806819c009d4|1    |17    |0              |
|0bbb30be-a832-4742-99c0-c0c3f2b14deb|1    |1     |0              |
|9a4440d5-c259-408a-926a-592c0d9666f6|1    |4     |0              |
|ceffe698-15ac-453f-8d18-1db7db106aa8|1    |1     |0              |
|9ba4aa1a-1073-4e49-b138-6c1d888f1a42|1    |4     |0              |
|3182ddd2-3868-4087-a2cf-5da0819f7453|1    |3     |0              |
|836a3986-ff77-4bbb-9f1a-a55f885bdf65|1    |1     |0              |
|2a3f660e-47ea-4938-a383-a4d57f7356ab|1    |4     |0              |
|25c226ec-7335-42c9-bf24-aa7164f120cc|1    |3     |0              |
|0b343527-17ac-4bb9-a7c1-898cab36818a|1    |1     |0              |
|5cfb194c-f0f6-4111-ae19-d75820c29adb|1    |1     |0              |
|0ba375f2-9170-4e8b-9888-e4e406dbd7c3|1    |1   

In [24]:
session_summary.filter(col("purchase_events") > 0).count()

629560

In [25]:
df.groupBy("category_code") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(20, truncate=False)

[Stage 95:=====================================================>  (41 + 2) / 43]

+--------------------------------+--------+
|category_code                   |count   |
+--------------------------------+--------+
|NULL                            |13515609|
|electronics.smartphone          |11507231|
|electronics.clocks              |1311033 |
|computers.notebook              |1137623 |
|electronics.video.tv            |1113750 |
|electronics.audio.headphone     |1100188 |
|appliances.kitchen.refrigerators|887755  |
|appliances.kitchen.washer       |869404  |
|appliances.environment.vacuum   |801670  |
|apparel.shoes                   |763901  |
|auto.accessories.player         |470208  |
|computers.desktop               |424242  |
|apparel.shoes.keds              |410304  |
|furniture.bedroom.bed           |358453  |
|electronics.tablet              |316735  |
|electronics.audio.subwoofer     |313664  |
|furniture.living_room.cabinet   |301410  |
|construction.tools.drill        |279948  |
|electronics.telephone           |247628  |
|auto.accessories.videoregister 

In [26]:
purchases.groupBy("category_code") \
    .agg(
        count("*").alias("purchase_events"),
        spark_sum("price").alias("revenue")
    ) \
    .orderBy(col("revenue").desc()) \
    .show(20, truncate=False)

[Stage 98:=====================================================>  (41 + 2) / 43]

+--------------------------------+---------------+--------------------+
|category_code                   |purchase_events|revenue             |
+--------------------------------+---------------+--------------------+
|electronics.smartphone          |338018         |1.5704962336999914E8|
|NULL                            |173425         |2.2924937129999954E7|
|computers.notebook              |15590          |8979887.25          |
|electronics.video.tv            |21565          |8423407.859999994   |
|electronics.clocks              |17906          |4818305.469999999   |
|appliances.kitchen.washer       |16148          |4658646.590000002   |
|appliances.kitchen.refrigerators|11218          |3830077.01          |
|electronics.audio.headphone     |30503          |3539126.9900000007  |
|appliances.environment.vacuum   |12378          |1716425.4099999995  |
|electronics.tablet              |5603           |1610973.7799999998  |
|computers.desktop               |3232           |1116821.980000

In [27]:
df.groupBy("brand") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(20, truncate=False)

[Stage 101:====================================================>  (41 + 2) / 43]

+--------+-------+
|brand   |count  |
+--------+-------+
|NULL    |6113008|
|samsung |5282775|
|apple   |4122554|
|xiaomi  |3083763|
|huawei  |1111205|
|lucente |655861 |
|lg      |562404 |
|bosch   |557090 |
|oppo    |482887 |
|sony    |456644 |
|acer    |428153 |
|cordiant|368277 |
|artel   |338203 |
|lenovo  |338103 |
|hp      |305242 |
|indesit |290101 |
|dauscher|289853 |
|philips |282699 |
|redmond |279734 |
|respect |271167 |
+--------+-------+
only showing top 20 rows


In [28]:
purchases.groupBy("brand") \
    .agg(
        count("*").alias("purchase_events"),
        spark_sum("price").alias("revenue")
    ) \
    .orderBy(col("revenue").desc()) \
    .show(20, truncate=False)

[Stage 104:====================================================>  (41 + 2) / 43]

+--------+---------------+-------------------+
|brand   |purchase_events|revenue            |
+--------+---------------+-------------------+
|apple   |142873         |1.112092688200001E8|
|samsung |172896         |4.640753261000007E7|
|xiaomi  |56616          |9194033.289999995  |
|NULL    |58214          |8540601.030000001  |
|huawei  |23501          |4883421.739999999  |
|acer    |6882           |3576719.5200000014 |
|lg      |8727           |3387887.9600000004 |
|lucente |11578          |3124113.370000002  |
|sony    |6729           |2478196.6799999997 |
|oppo    |10891          |2412959.7600000002 |
|lenovo  |4578           |1752638.5299999993 |
|indesit |5024           |1250060.95         |
|bosch   |5705           |1248729.0899999999 |
|hp      |3596           |1227215.99         |
|artel   |6124           |1034152.4099999999 |
|asus    |1850           |970019.39          |
|beko    |3961           |963940.5899999999  |
|haier   |3072           |892047.6899999998  |
|dauscher|353

In [29]:
df.filter(col("event_type") == "view") \
  .groupBy("product_id") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(20)

[Stage 107:====================================================>  (41 + 2) / 43]

+----------+------+
|product_id| count|
+----------+------+
|   1004856|419287|
|   1004767|378777|
|   1005115|327715|
|   1004249|207422|
|   1004833|203018|
|   1005105|197930|
|   1004870|190435|
|   1002544|179249|
|   4804056|179092|
|   5100816|164608|
|   1004741|151176|
|   1004836|143985|
|   1004873|137342|
|   1004739|137295|
|   1005160|132726|
|   1002524|117273|
|   1004785|116315|
|   1002633|109305|
|   1004750|108847|
|   1005100|108469|
+----------+------+
only showing top 20 rows


In [30]:
purchases.groupBy("product_id") \
    .agg(
        count("*").alias("purchase_events"),
        spark_sum("price").alias("revenue")
    ) \
    .orderBy(col("revenue").desc()) \
    .show(20)

[Stage 110:====================================================>  (41 + 2) / 43]

+----------+---------------+--------------------+
|product_id|purchase_events|             revenue|
+----------+---------------+--------------------+
|   1005115|          12543|1.2406807349999985E7|
|   1005105|           7293|1.0239248680000003E7|
|   1004249|           9090|   6730112.919999998|
|   1005135|           3218|   5567806.639999998|
|   1004767|          21806|   5430723.430000015|
|   1002544|          10549|   4855702.390000005|
|   1004856|          28944|  3798956.4199999925|
|   1002524|           6633|  3538299.1199999982|
|   1003317|           3220|          3051294.26|
|   1004870|          10615|  3027098.0499999993|
|   1003306|           4375|          2552587.18|
|   1004237|           2249|  2449741.8799999994|
|   1004258|           3152|  2332809.7099999995|
|   1004873|           6053|   2269184.629999999|
|   1004833|          12697|   2184501.959999999|
|   1005106|           1316|  2000494.0499999993|
|   4804056|          12381|  1990026.6800000027|


In [31]:
event_counts = (
    df.groupBy("event_type")
      .count()
      .collect()
)

event_counts

[Row(event_type='purchase', count=742849),
 Row(event_type='view', count=40779399),
 Row(event_type='cart', count=926516)]

In [32]:
funnel = (
    df.groupBy("event_type")
      .count()
      .withColumnRenamed("count", "events")
      .orderBy(col("events").desc())
)

funnel.show()

[Stage 116:====================================================>  (41 + 2) / 43]

+----------+--------+
|event_type|  events|
+----------+--------+
|      view|40779399|
|      cart|  926516|
|  purchase|  742849|
+----------+--------+



In [33]:
total_events = df.count()

funnel.withColumn(
    "percentage",
    (col("events") / total_events) * 100
).show()

[Stage 122:====================================================>  (41 + 2) / 43]

+----------+--------+------------------+
|event_type|  events|        percentage|
+----------+--------+------------------+
|      view|40779399| 96.06734132470854|
|      cart|  926516| 2.182668970055288|
|  purchase|  742849|1.7499897052361761|
+----------+--------+------------------+



In [35]:
total_users = df.select("user_id").distinct().count()
purchasing_users = purchases.select("user_id").distinct().count()

purchase_participation = (purchasing_users / total_users) * 100

print("Total users:", total_users)
print("Purchasing users:", purchasing_users)
print(
    "User purchase participation (%):",
    f"{purchase_participation:.2f}"
)

[Stage 143:====================================================>  (41 + 2) / 43]

Total users: 3022290
Purchasing users: 347118
User purchase participation (%): 11.49


In [36]:
df.select("price").summary(
    "count",
    "mean",
    "stddev",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show()

[Stage 151:>                                                        (0 + 1) / 1]

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|         42448764|
|   mean|290.3236606848809|
| stddev|358.2691553394021|
|    min|              0.0|
|    25%|            65.97|
|    50%|           162.93|
|    75%|           358.57|
|    max|          2574.07|
+-------+-----------------+



In [37]:
purchases.select("price").summary(
    "count",
    "mean",
    "stddev",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show()

[Stage 152:====================================================>  (41 + 2) / 43]

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|            742849|
|   mean|309.56156940373995|
| stddev| 348.8732405995006|
|    min|              0.77|
|    25%|             87.45|
|    50%|            179.84|
|    75%|            375.04|
|    max|           2574.07|
+-------+------------------+



In [38]:
df.select(
    spark_sum(when(col("user_id").isNull(), 1).otherwise(0)).alias("null_user_id"),
    spark_sum(when(col("product_id").isNull(), 1).otherwise(0)).alias("null_product_id"),
    spark_sum(when(col("category_id").isNull(), 1).otherwise(0)).alias("null_category_id"),
    spark_sum(when(col("event_time").isNull(), 1).otherwise(0)).alias("null_event_time"),
    spark_sum(when(col("price").isNull(), 1).otherwise(0)).alias("null_price")
).show()

[Stage 155:====================================================>  (41 + 2) / 43]

+------------+---------------+----------------+---------------+----------+
|null_user_id|null_product_id|null_category_id|null_event_time|null_price|
+------------+---------------+----------------+---------------+----------+
|           0|              0|               0|              0|         0|
+------------+---------------+----------------+---------------+----------+



In [39]:
valid_events = ["view", "cart", "purchase"]

df.filter(~col("event_type").isin(valid_events)) \
  .groupBy("event_type") \
  .count() \
  .show()

[Stage 158:====================================================>  (41 + 2) / 43]

+----------+-----+
|event_type|count|
+----------+-----+
+----------+-----+



In [3]:
from pyspark.sql.functions import col

purchases = df.filter(col("event_type") == "purchase")

print("Purchase events:", purchases.count())

[Stage 5:======================================================>  (41 + 2) / 43]

Purchase events: 742849


In [4]:
purchase_duplicates = (
    purchases
    .groupBy(
        "user_id",
        "user_session",
        "product_id",
        "event_time",
        "price"
    )
    .count()
    .filter(col("count") > 1)
)

purchase_duplicates_count = purchase_duplicates.count()

print("Repeated purchase records:", purchase_duplicates_count)

[Stage 8:======================================================>  (41 + 2) / 43]

Repeated purchase records: 76


## Exploration checklist
1. Dataset dimensions
2. Schema and sample records
3. Event-type distribution
4. Unique users, products, sessions
5. Missing values
6. Price statistics
7. Event-time range
8. Initial observations
